# SentinelRUL: Forecast Pretraining

Trains the shared GRU backbone on next-N-cycle sensor prediction before multitask fine tuning.
Run this on Kaggle with GPU accelerator. Download `forecast_best.pt` from `/kaggle/working/` when finished.

In [ ]:
!pip install -q torch numpy pandas scikit-learn matplotlib

In [ ]:
import os
import math
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

print(f'torch {torch.__version__}  cuda: {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using device: {DEVICE}')

## Data Configuration

Point `DATA_DIR` at the Kaggle dataset input folder. Add the NASA C-MAPSS dataset to your notebook
via *Add Data* and update the path below if the slug differs.

In [ ]:
DATA_DIR = '/kaggle/input/nasa-cmapss-data'

ALL_COLS = ['engine_id', 'cycle', 'os1', 'os2', 'os3'] + [f's{i}' for i in range(1, 22)]

DROP_SENSORS = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
SENSOR_COLS  = [c for c in ALL_COLS if c.startswith('s') and c not in DROP_SENSORS]

WINDOW_SIZE      = 30
FORECAST_HORIZON = 5
RUL_CLIP         = 125
VAL_ENGINES      = 20
SEED             = 42

print(f'{len(SENSOR_COLS)} sensors kept: {SENSOR_COLS}')

## Data Loading and Preprocessing

In [ ]:
def load_fd001(data_dir):
    path = os.path.join(data_dir, 'train_FD001.txt')
    df = pd.read_csv(path, sep=r'\s+', header=None, names=ALL_COLS)
    return df


def add_rul_labels(df, rul_clip=RUL_CLIP):
    max_cycle = df.groupby('engine_id')['cycle'].max().rename('max_cycle')
    df = df.join(max_cycle, on='engine_id')
    df['rul'] = (df['max_cycle'] - df['cycle']).clip(upper=rul_clip)
    df.drop(columns=['max_cycle'], inplace=True)
    return df


def fit_scaler(df):
    mins = df[SENSOR_COLS].min()
    maxs = df[SENSOR_COLS].max()
    return mins, maxs


def normalize(df, mins, maxs):
    df = df.copy()
    denom = (maxs - mins).replace(0, 1)
    df[SENSOR_COLS] = (df[SENSOR_COLS] - mins) / denom
    return df


def prepare(data_dir):
    raw = load_fd001(data_dir)
    df  = add_rul_labels(raw.copy())
    mins, maxs = fit_scaler(df)
    df  = normalize(df, mins, maxs)
    return df, mins, maxs


full_df, mins, maxs = prepare(DATA_DIR)
print(f'rows: {len(full_df)}  engines: {full_df["engine_id"].nunique()}')
full_df.head(3)